# ADI sinwave convergence in z

Spatial convergence study with **fixed** `dx`, `dy` (`n_x = n_y = 4`) and varying `n_z`.
Uses fixed `adi.dt = 0.01`, runs the traveling `sinwave` IC for **1 s**, and reports the
**L1 error** of `Ex` in **(z, t)** at the mid-(x, y) line:

$$\|e\|_1 = \sum_{i,j} |E_x^{\mathrm{num}}(z_i, t_j) - E_x^{\mathrm{exact}}(z_i, t_j)|\,\Delta z\,\Delta t$$

Run from the repo root (needs a built `main3d.gnu.ex`):
```bash
make -j
```

In [ ]:
import json
import shutil
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# --- problem setup (keep in sync with `inputs`) ---
EXE = (Path.cwd() / "main3d.gnu.ex").resolve()
INPUTS = Path("inputs")
NX, NY = 4, 4
# Vary n_z only; dx = dy = 1/4 are fixed. With dt=0.01 only modest n_z is stable
# (ADI blows up once dz is small enough). Extend toward 128+ with smaller dt.
NZ_LIST = [4, 8, 16, 32, 64, 128, 256, 512, 1024]

DT = 5e-13
MAX_STEP = 2
T_END = MAX_STEP * DT

IC_AMPLITUDE = 0.01
IC = "sinwave"          # "sinwave" or "gaussianpulse"
IC_DIR = 2              # +z propagation
IC_POL = 0              # Ex
IC_WAVELENGTH = -1      # <0 => one wavelength along ic_dir (domain length)
PULSE_CENTER = 0.75     # envelope peak along ic_dir (gaussianpulse)
PULSE_SIGMA = 0.05      # envelope sigma; <0 => 0.1 * domain length along ic_dir
PEC_NORMAL = -1         # -1: periodic; else PEC on faces normal to this axis

C_LIGHT = 2.99792458e8
COMPONENTS = ["Ex", "Ey", "Ez", "Bx", "By", "Bz"]
COMP_IDX = IC_POL

OUTPUT_ROOT = Path("plt_conv")
STABILITY_THRESHOLD = 10.0 * IC_AMPLITUDE  # flag likely blow-up

In [ ]:
def list_steps(output_dir: Path):
    metas = sorted(output_dir.glob("step_*_meta.json"))
    return [(json.loads(p.read_text())["step"], p) for p in metas]


def load_step(meta_path: Path):
    meta = json.loads(meta_path.read_text())
    bin_path = meta_path.with_name(meta_path.name.replace("_meta.json", "_fields.bin"))
    fields = np.fromfile(bin_path, dtype=np.float64).reshape(meta["shape"])
    return fields, meta


def load_series(output_dir: Path):
    series = []
    for step, meta_path in list_steps(output_dir):
        fields, meta = load_step(meta_path)
        series.append((meta["step"], meta["time"], fields, meta))
    return series


def coord_axes(meta):
    nx, ny, nz, _ = meta["shape"]
    prob_lo = np.array(meta["prob_lo"])
    prob_hi = np.array(meta["prob_hi"])
    dx = (prob_hi[0] - prob_lo[0]) / nx
    dy = (prob_hi[1] - prob_lo[1]) / ny
    dz = (prob_hi[2] - prob_lo[2]) / nz
    x = prob_lo[0] + (np.arange(nx) + 0.5) * dx
    y = prob_lo[1] + (np.arange(ny) + 0.5) * dy
    z = prob_lo[2] + (np.arange(nz) + 0.5) * dz
    return x, y, z, dx, dy, dz


def ic_wavenumber(meta):
    prob_lo = np.array(meta["prob_lo"])
    prob_hi = np.array(meta["prob_hi"])
    domain_len = prob_hi[IC_DIR] - prob_lo[IC_DIR]
    wavelength = domain_len if IC_WAVELENGTH < 0 else IC_WAVELENGTH
    return 2.0 * np.pi / wavelength, domain_len


def pulse_params(meta):
    prob_lo = np.array(meta["prob_lo"])
    prob_hi = np.array(meta["prob_hi"])
    _, domain_len = ic_wavenumber(meta)
    sigma = PULSE_SIGMA if PULSE_SIGMA > 0 else 0.1 * domain_len
    return float(PULSE_CENTER), float(sigma)


def is_periodic_along_ic():
    return PEC_NORMAL < 0 or PEC_NORMAL != IC_DIR


def periodic_gaussian_pulse(xi, center, sigma, k, domain_len):
    """Sum Gaussian carrier over periodic images along ic_dir."""
    n_pad = int(np.ceil(3.0 * sigma / domain_len)) + 2
    offsets = domain_len * np.arange(-n_pad, n_pad + 1)
    eta = np.asarray(xi)[..., np.newaxis] - np.asarray(center)[..., np.newaxis] - offsets
    return np.sum(np.exp(-0.5 * (eta / sigma) ** 2) * np.cos(k * eta), axis=-1)


def exact_ex(z, t, meta):
    """Exact E_pol along ic_dir for sinwave or gaussianpulse (ic_pol=Ex, ic_dir=+z here)."""
    if IC == "sinwave":
        k, _ = ic_wavenumber(meta)
        return IC_AMPLITUDE * np.sin(k * (z - C_LIGHT * t))

    if IC == "gaussianpulse":
        x0, sigma = pulse_params(meta)
        k, domain_len = ic_wavenumber(meta)
        center = x0 + C_LIGHT * t
        if is_periodic_along_ic():
            g = periodic_gaussian_pulse(z, center, sigma, k, domain_len)
        else:
            eta = z - center
            g = np.exp(-0.5 * (eta / sigma) ** 2) * np.cos(k * eta)
        return IC_AMPLITUDE * g

    raise ValueError(f"Unsupported IC={IC!r}; use 'sinwave' or 'gaussianpulse'")


def run_sim(nz: int, output_dir: Path, rebuild: bool = False):
    if rebuild or not EXE.exists():
        subprocess.run(["make", "-j"], cwd=EXE.parent, check=True)

    if output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        str(EXE),
        str(INPUTS),
        f"adi.n_cells={NX} {NY} {nz}",
        "adi.cfl=-1",
        f"adi.dt={DT}",
        f"adi.max_step={MAX_STEP}",
        "adi.plot_int=1",
        "adi.plot_format=numpy",
        f"adi.output_dir={output_dir}",
        f"adi.ic={IC}",
        f"adi.ic_amplitude={IC_AMPLITUDE}",
        f"adi.ic_dir={IC_DIR}",
        f"adi.ic_pol={IC_POL}",
        f"adi.ic_wavelength={IC_WAVELENGTH}",
        f"adi.pec_normal={PEC_NORMAL}",
    ]
    if IC == "gaussianpulse":
        cmd.extend(
            [
                f"adi.pulse_center={PULSE_CENTER}",
                f"adi.pulse_sigma={PULSE_SIGMA}",
            ]
        )
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)


def l1_error_zt(output_dir: Path):
    series = load_series(output_dir)
    if not series:
        raise FileNotFoundError(f"No dumps in {output_dir}")

    _, _, fields0, meta0 = series[0]
    x, y, z, dx, dy, dz = coord_axes(meta0)
    ix, iy = len(x) // 2, len(y) // 2

    times = np.array([meta["time"] for _, _, _, meta in series])
    sim = np.array([fields[ix, iy, :, COMP_IDX] for _, _, fields, _ in series])
    max_abs = float(np.max(np.abs(sim)))

    zz, tt = np.meshgrid(z, times, indexing="xy")
    exact = exact_ex(zz, tt, meta0)

    dt_eff = float(np.mean(np.diff(times))) if len(times) > 1 else DT
    l1 = float(np.sum(np.abs(sim - exact)) * dz * dt_eff)
    l1_mean = float(np.mean(np.abs(sim - exact)))

    return {
        "nz": meta0["shape"][2],
        "dz": dz,
        "dx": dx,
        "dy": dy,
        "l1": l1,
        "l1_mean": l1_mean,
        "max_abs": max_abs,
        "stable": max_abs < STABILITY_THRESHOLD,
        "z": z,
        "times": times,
        "sim": sim,
        "exact": exact,
        "output_dir": output_dir,
    }

In [ ]:
results = []
for nz in NZ_LIST:
    out_dir = OUTPUT_ROOT / f"nz_{nz}"
    print(f"\n=== n_cells = {NX} {NY} {nz} (dz = {1.0 / nz:.4g}) ===")
    try:
        run_sim(nz, out_dir)
        res = l1_error_zt(out_dir)
    except subprocess.CalledProcessError as exc:
        print(f"  simulation failed (exit {exc.returncode}) — likely unstable at this dz for dt={DT}")
        continue
    results.append(res)
    flag = "" if res["stable"] else "  ** UNSTABLE / blown up **"
    print(
        f"  L1(z,t) = {res['l1']:.3e}   "
        f"mean|err| = {res['l1_mean']:.3e}   "
        f"max|Ex| = {res['max_abs']:.3e}   "
        f"dx = {res['dx']:.3g}, dy = {res['dy']:.3g}, dz = {res['dz']:.3g}"
        f"{flag}"
    )

nz_arr = np.array([r["nz"] for r in results])
dz_arr = np.array([r["dz"] for r in results])
l1_arr = np.array([r["l1"] for r in results])

print("\nSummary")
for r in results:
    tag = "ok" if r["stable"] else "UNSTABLE"
    print(f"  [{tag}] nz={r['nz']:4d}  dz={r['dz']:.4g}  L1={r['l1']:.3e}")

In [ ]:
if not results:
    raise RuntimeError("No successful runs")

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)

ax = axes[0]
ax.loglog(dz_arr, l1_arr, "o-", lw=1.5, label="simulation")
ref_o2 = l1_arr[0] * (dz_arr / dz_arr[0]) ** 2
ax.loglog(dz_arr, ref_o2, "k--", lw=1.0, alpha=0.7, label=r"$O(\Delta z^2)$")
for nz, dz, l1 in zip(nz_arr, dz_arr, l1_arr):
    ax.annotate(f"nz={nz}", (dz, l1), textcoords="offset points", xytext=(4, 4), fontsize=8)
ax.set_xlabel(r"$\Delta z$")
ax.set_ylabel(r"$L_1$ error in $(z,t)$")
ax.set_title(f"{COMPONENTS[COMP_IDX]} ({IC}) at mid-(x,y), dt={DT}, t_end={T_END}")
ax.legend(loc="best")
ax.grid(True, which="both", alpha=0.3)

ax = axes[1]
if len(dz_arr) > 1:
    orders = np.log(l1_arr[:-1] / l1_arr[1:]) / np.log(dz_arr[:-1] / dz_arr[1:])
    mids = 0.5 * (dz_arr[:-1] + dz_arr[1:])
    ax.plot(mids, orders, "s-")
    ax.axhline(0, color="k", lw=0.5)
    ax.set_xscale("log")
    ax.set_xlabel(r"$\Delta z$")
    ax.set_ylabel("observed order")
    ax.set_title("p ≈ log(L1_i/L1_{i+1}) / log(dz_i/dz_{i+1})")
    ax.grid(True, which="both", alpha=0.3)
else:
    ax.axis("off")

plt.show()

In [ ]:
# Space-time slice: simulation vs exact (finest grid)
fine = results[-1]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), constrained_layout=True)
extent = [fine["z"][0], fine["z"][-1], fine["times"][0], fine["times"][-1]]

for ax, data, title in zip(
    axes,
    [fine["sim"], fine["exact"], fine["sim"] - fine["exact"]],
    ["simulation", "exact", "error"],
):
    vmax = np.max(np.abs(data))
    im = ax.imshow(
        data,
        origin="lower",
        aspect="auto",
        extent=extent,
        cmap="RdBu_r" if title != "error" else "coolwarm",
        vmin=-vmax,
        vmax=vmax,
    )
    ax.set_xlabel("z")
    ax.set_ylabel("t")
    ax.set_title(f"{title}  (nz={fine['nz']})")
    fig.colorbar(im, ax=ax, fraction=0.046)

plt.show()